## ___Updating the photosynthetic pathways___
--------------------

In [1]:
!python --version

Python 3.13.7


In [1]:
import numpy as np
import pandas as pd

In [2]:
# https://www.tern.org.au/news/news-photosynthetic-pathways/
# https://portal.tern.org.au/metadata/TERN/1e16257d-57ae-48dd-bbad-5701e72a9f6d
# TERN is an Australia specific dataset
tern = pd.read_csv(r"../../data/chapter2/TERN/Photosynthetic_Pathways_of_Plants_TERN_v2_19092024.csv", encoding="latin1", 
        usecols=["genus", "speciesEpithet", "family", "photosyntheticPathway_confirmed", "photosyntheticPathway_inferred", "photosyntheticPathway_combined"], na_values='U')#, index_col=["genus", "speciesEpithet"])
tern_meta = pd.read_excel(r"../../data/chapter2/TERN/metadata_Photosynthetic_Pathways_of_Plants_TERN_v2_2024.xlsx", sheet_name="Data_Descriptor")

# extra spaces
tern.loc[:, "genus"] = tern.genus.str.strip()
tern.loc[:, "speciesEpithet"] = tern.speciesEpithet.str.strip()

# in TRY, photosynthesis pathway is trait id 22
try_photo = pd.read_csv(r"../../data/chapter2/TRY/photosynthetic_pathways.txt", delimiter='\t', encoding="latin1", low_memory=False, decimal='.', usecols=["Dataset", "SpeciesName",
                        "AccSpeciesName", "OrigValueStr", "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"]) # records where TraitId is NaN are mostly messy metadata rows

# extra spaces 
try_photo.loc[:, "SpeciesName"] = try_photo.SpeciesName.str.strip()
try_photo.loc[:, "OrigValueStr"] = try_photo.OrigValueStr.str.upper().str.replace('.', '')
# messy photosynthetic pathway information
PHOTOSYNTHETIC_PATHWAY_TYPES = { # try to make this as less messy as possible!!
    "C3": "C3",
    "C3?": "C3?",
    "C4": "C4",
    "C4?": "C4?",
    "CAM": "CAM",
    "CAM?": "CAM?",
    "C3/C4": "C3/C4",
    "C3C4": "C3/C4",
    "C3/CAM": "C3/CAM",
    "C3-CAM": "C3/CAM",
    "C4/CAM": "C4/CAM",
    "C4-CAM": "C4/CAM",
    "C3/C4/CAM": "C3/C4/CAM",
    "3": "C3"
}

try_photo.loc[:, "OrigValueStr"] = try_photo.OrigValueStr.replace(PHOTOSYNTHETIC_PATHWAY_TYPES) # clean up the irregularities in the column
subset_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_categorical.csv")

In [3]:
subset_categorical.F00043.unique()

array(['C3', nan], dtype=object)

In [10]:
groot = pd.read_csv(r"../../data/chapter2/GRooTFullVersion.csv", encoding="latin1", low_memory=False)
groot.insert(loc=0, column="binominal", value=groot.genus.str.capitalize().str.strip() + ' ' + groot.species.str.lower().str.strip())

In [24]:
groot.genus.str.capitalize().str.strip() + ' ' + groot.species.str.lower().str.strip()

0                  NaN
1                  NaN
2                  NaN
3                  NaN
4                  NaN
              ...     
114217    Inula conyza
114218    Inula conyza
114219    Inula conyza
114220    Inula conyza
114221    Inula conyza
Length: 114222, dtype: object

In [31]:
groot_photo = groot.query("not photosyntheticPathway.isna() and not binominal.isna()").loc[:, ["binominal", "photosyntheticPathway"]].drop_duplicates().reset_index(drop=True)
groot_photo

,binominal,photosyntheticPathway
0,Agropyron cristatum,C3
1,Artemisia tridentata,C3
2,Elymus elymoides,C3
3,Acer saccharum,C3
4,Dacrydium cupressinum,C3
...,...,...
4526,Senecio umbrosus,C3
4527,Tephroseris tenuifolia,C3
4528,Veronica aphylla,C3
4529,Vicia onobrychioides,C3


In [34]:
pd.merge(left=subset_categorical.query("F00043.isna()"), left_on="binominal", right=groot_photo, right_on="binominal")

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004,photosyntheticPathway
0,Rhaphiolepis indica,Rhaphiolepis,indica,Rosaceae,Rosales,NaN,AM,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3
1,Aesculus hippocastanum,Aesculus,hippocastanum,Sapindaceae,Sapindales,NaN,NaN,Valverde et al (unpublished),C3


In [8]:
pd.merge(left=subset_categorical, left_on="binominal", right=try_photo, right_on="AccSpeciesName", how="inner").drop_duplicates(subset=["AccSpeciesName", "OrigValueStr"]).OrigValueStr.unique()

array(['C3', 'C4', 'C3?', 'UNKNOWN'], dtype=object)

In [ ]:
pd.merge()